In [1]:
import os
import base64
from pathlib import Path
from typing import TypedDict, List, Optional
from dotenv import load_dotenv

from langgraph.graph import START, END, StateGraph
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

D:\code\LangChain-lesson\My-test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL=os.getenv("DASHSCOPE_BASE_URL")
if not DASHSCOPE_API_KEY or DASHSCOPE_API_KEY == "your_dashscope_api_key_here":
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")
# 初始化模型（图像处理需要支持视觉的模型）
model = init_chat_model(
    model="qwen-vl-max",
    model_provider="openai",
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)
try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()
IMAGES_DIR = SCRIPT_DIR / "images"

# 辅助函数

In [3]:
# 本地图片编码为 base64
def encode_image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        image_bytes = image_file.read()
        return base64.standard_b64encode(image_bytes).decode("utf-8")
# 根据文件扩展名获取 MINE 类型
def get_mine_type(image_path: str) -> str:
    ext = Path(image_path).suffix.lower()
    mine_type = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    return mine_type.get(ext, "image/jpeg")
# 创建图像内容块
def create_image_content(image_path: str) -> dict:
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件 {image_path} 不存在")

    image_base64 = encode_image_to_base64(image_path)
    mine_type = get_mine_type(image_path)

    return {
        "type": "image_url",
        "image_url": {
            "url": f"data:{mine_type};base64,{image_base64}"
        }
    }
# 检查图片是否存在
def check_image_exists(filename: str) -> str:
    image_path = IMAGES_DIR / filename
    if not image_path.exists():
        print(f"图片 {filename} 不存在")
        return None
    return str(image_path)

# 文本 + 图像混合输入

In [4]:
def example_1_text_and_image():
    """
    处理文本和图像的混合输入
    """
    print("\n"+"="*60)
    print("处理文本和图像的混合输入")
    print("="*60)
    image_path = check_image_exists("chart.png")
    if not image_path:
        print("请先将图片 chart.png 放入 images 目录下")
        return None
    # 创建混合内容消息
    content = [
        {"type": "text", "text": """以下是我们公司的销售数据：

**2024年第一季度销售报告**
- 1月: 150万
- 2月: 180万
- 3月: 220万

请结合图表分析：
1. 数据趋势如何？
2. 与图表显示的趋势是否一致？
3. 你有什么建议？"""},
        create_image_content(image_path)
    ]
    message = HumanMessage(content=content)
    response = model.invoke([message])
    print("\n分析结果：")
    print(response.content)

    return response.content

# 多图像对比分析

In [5]:
def example_2_multi_image():
    """
    对比多张图片
    """
    print("\n"+"="*60)
    print("多图像对比分析")
    print("="*60)

    image1_path = check_image_exists("image1.jpg")
    image2_path = check_image_exists("image2.jpg")

    if not image1_path or not image2_path:
        print("请先将图片 image1.jpg 和 image2.jpg 放入 images 目录下")
        return None

    content = [
        {"type": "text", "text": "请比较两张图片，并给出你的建议"},
        create_image_content(image1_path),
        create_image_content(image2_path)
    ]
    message = HumanMessage(content=content)
    response = model.invoke([message])
    print("\n分析结果：")
    print(response.content)
    return response.content

# 使用 Langgraph 处理混合模态

In [6]:
class MultimodalState(TypedDict):
    """混合模态状态"""
    text_input: str
    image_paths: List[str]
    analysis_result: Optional[str]
    summary: Optional[str]
def example_3_langgraph_multimodal():
    print("\n"+"="*60)
    print("使用 Langgraph 处理混合模态")
    print("="*60)

    image_path = check_image_exists("sample.jpg")
    if not image_path:
        print("请先将图片 sample.jpg 放入 images 目录下")
        return None

    # 节点函数
    def analyze_content(state: MultimodalState) -> MultimodalState:
        """分析混合内容"""
        print("正在分析内容...")
        content = [
            {"type": "text", "text": state["text_input"]}
        ]
        for img_path in state["image_paths"]:
            if os.path.exists(img_path):
                content.append(create_image_content(img_path))

        message = HumanMessage(content=content)
        response = model.invoke([message])
        state["analysis_result"] = response.content
        return state
    def summarize(state: MultimodalState) -> MultimodalState:
        """总结分析结果"""
        print("正在总结分析结果...")
        message = HumanMessage(
            content=f"请用3句话总结以下分析：\n\n{state['analysis_result']}"
        )
        repsonse = model.invoke([message])
        state["summary"] = repsonse.content
        return state

    # 构造图
    graph = StateGraph(MultimodalState)
    #  添加节点
    graph.add_node("analyze", analyze_content)
    graph.add_node("summarize", summarize)
    # 添加边
    graph.add_edge(START, "analyze")
    graph.add_edge("analyze", "summarize")
    graph.add_edge("summarize", END)
    # 编译
    workflow = graph.compile()
    # 测试
    initial_state = {
        "text_input": "请详细描述这张图片，包括主要内容、色彩和氛围。",
        "image_paths": [image_path],
        "analysis_result": None,
        "summary": None
    }

    result = workflow.invoke(initial_state)

    print("\n详细分析：")
    print(result["analysis_result"])
    print("\n总结：")
    print(result["summary"])

    return result


# 基于图像的交互式问答

In [7]:
def example_4_interactive_qa():
    """
    基于图像的交互式问答
    """
    print("\n" + "=" * 60)
    print("交互式图像问答（演示）")
    print("=" * 60)

    image_path = check_image_exists("sample.jpg")
    if not image_path:
        print("请在 images/ 目录下放置 sample.jpg")
        print("跳过此示例")
        return None

    # 模拟问答流程
    questions = [
        "这张图片展示了什么？",
        "图片中有哪些颜色？",
        "你觉得这张图片是在什么场景下拍摄的？"
    ]

    messages = []

    # 首先发送图片
    initial_content = [
        {"type": "text", "text": "我将基于这张图片问你一些问题。"},
        create_image_content(image_path)
    ]
    messages.append(HumanMessage(content=initial_content))

    print(f"📷 已加载图片: {image_path}\n")

    for q in questions:
        print(f"❓ 问题: {q}")

        messages.append(HumanMessage(content=q))
        response = model.invoke(messages)
        messages.append(response)

        print(f"💬 回答: {response.content}\n")

    print("💡 提示: 实际应用中可以使用 input() 实现真正的交互式问答")

    return messages

In [9]:
def main():
    print("\n"+"="*70)
    print("混合多模态处理")
    print("="*70)
    # 创建图片目录
    IMAGES_DIR.mkdir(exist_ok=True)

    example_1_text_and_image()
    example_2_multi_image()
    example_3_langgraph_multimodal()
    example_4_interactive_qa()

In [10]:
if __name__ == "__main__":
    main()


混合多模态处理

处理文本和图像的混合输入

分析结果：
根据您提供的2024年第一季度销售数据和图表，我将从以下三个方面进行分析：

---

### **1. 数据趋势如何？**

从数据来看：
- 1月销售额为 **150万元**
- 2月增长至 **180万元**，环比增长 **20%**
- 3月进一步上升至 **220万元**，环比增长约 **22.2%**

整体呈现 **持续上升趋势**，且增速略有加快。这表明公司在第一季度的市场表现稳步向好，可能得益于市场需求增加、营销策略有效或产品推广力度加大。

---

### **2. 与图表显示的趋势是否一致？**

是的，**图表与数据完全一致**。

- 图表为柱状图，清晰地展示了每个月的销售额。
- 每根柱子的高度逐月递增：1月（蓝色）< 2月（绿色）< 3月（红色）
- 数值标注准确无误（150万、180万、220万）
- Y轴刻度合理，便于直观比较

因此，**图表真实、准确地反映了销售数据的增长趋势**，视觉传达清晰，没有误导性。

---

### **3. 建议**

基于当前良好的增长态势，提出以下几点建议：

#### ✅ **继续保持并优化增长动力**
- 分析3月份增长加速的原因（如促销活动、新品上市、节日效应等），总结成功经验并复制到后续季度。
- 维持现有营销节奏，避免因短期成功而松懈。

#### ✅ **深入挖掘增长驱动因素**
- 对比去年同期数据，判断是行业整体回暖还是公司自身策略见效。
- 调研客户反馈，了解哪些产品或服务贡献最大，集中资源投入高潜力领域。

#### ✅ **设定第二季度目标**
- 一季度总销售额：550万元
- 若保持每月约20%的环比增长，预计二季度可实现更高目标（如：4月264万、5月317万、6月380万）
- 可据此制定更具挑战性的季度目标，并分解到团队和个人。

#### ✅ **加强数据分析与可视化**
- 建议未来增加同比、环比增长率标签在图表中，提升信息密度。
- 可引入折线图叠加趋势线，更直观展示增长轨迹。

---

### 📌 总结

> **结论**：2024年第一季度销售数据呈现稳健增长趋势，图表准确反映实际情况。建议乘势而上，深化分析，科学规划下一阶段战略，确保可持续发展。

如需，我可以协助制作下一阶段的预测模型或